In [ ]:
"""GHI dataset preprocessing guide

This notebook merges NSRDB station CSVs (skipping the first two rows and using the third row as header) for station codes from CYL_geo/CyL_geo.csv, saves per-station merged files to NSRDB/merged_files2, and adds a unified datetime "timestamp" column. It then filters CYL_GHI CSVs to the 2017Ã¢â‚¬â€œ2019 period, imputes mapped measurements from NSRDB except precipitation and toa, which are interpolated over time and edge-filled from their own station series, and writes the results to CYL_GHI/filtered_files. A final check lists CSVs containing blank or empty values.

Workflow details: the NSRDB merge uses the station_code values from the geography CSV, then creates timestamps from the date/time columns. CYL_GHI rows are filtered to the 2017Ã¢â‚¬â€œ2019 interval; mapped missing measurements are filled from the matching station's NSRDB data, while precipitation and toa use time interpolation with forward/backward fill at the edges. Cloud tcc is joined afterward; cyclic features are derived before tensor construction.
The tensor archive stores the full tensor under the key 'tensor', plus 'features', 'feature_means', 'feature_stds', and 'normalized_features'. The final cell's filtered_tensor and target_tensor are in-memory views only.
To retrieve raw GHI from the saved archive, load saved_tensor = data['tensor'] and saved_features = data['features'].astype(str), then select saved_tensor[:, :, saved_features == 'GHI']; the result has shape [T, N, 1]. For load_windowed_drive_folds, pass saved_features == 'GHI' as output_mask_key.
Run cells in order. Set load_existing_npz in the tensor-building cell to reuse the saved tensor. Review station matches, timestamp coverage, and missing-value reports before using the results for forecasting.

Inputs: CYL_geo/CyL_geo.csv, NSRDB/*.csv, CYL_GHI/*.csv
Outputs: NSRDB/merged_files2/*_merged.csv, CYL_GHI/filtered_files/*_filtered.csv

"""

In [ ]:
import os
import glob
import pandas as pd

# Mount Google Drive when running in Colab
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
    base_dir = '/content/gdrive/MyDrive'
except Exception:
    base_dir = os.path.expanduser('~/MyDrive')

geo_path = os.path.join(base_dir, 'CYL_geo', 'CyL_geo.csv')
nsrdb_dir = os.path.join(base_dir, 'NSRDB')
merged_dir = os.path.join(nsrdb_dir, 'merged_files2')

In [ ]:
"""Load paths and station metadata, merge each station's NSRDB files, and add timestamps.

Input: station codes and NSRDB station CSVs. Output: merged station CSVs in NSRDB/merged_files2.
"""

if not os.path.exists(geo_path):
    raise FileNotFoundError(f'Geo file not found: {geo_path}')

os.makedirs(merged_dir, exist_ok=True)

geo_df = pd.read_csv(geo_path)
if 'station_code' not in geo_df.columns:
    raise KeyError('The file must contain a station_code column.')

station_codes = [str(code).strip() for code in geo_df['station_code'].dropna().unique() if str(code).strip()]

print(f'Found {len(station_codes)} station codes in {geo_path}')

for station_code in station_codes:
    pattern = os.path.join(nsrdb_dir, f'{station_code}_*.csv')
    files = sorted(glob.glob(pattern))

    if not files:
        print(f'No files found for {station_code}')
        continue

    frames = []
    for file_path in files:
        df = pd.read_csv(file_path, skiprows=2)
        frames.append(df)

    merged_df = pd.concat(frames, ignore_index=True, axis=0)

    if 'Solar Zenith Angle' in merged_df.columns:
        merged_df['solar_elev'] = 90 - pd.to_numeric(merged_df['Solar Zenith Angle'], errors='coerce')
        merged_df = merged_df.drop(columns=['Solar Zenith Angle'])

    output_path = os.path.join(merged_dir, f'{station_code}_merged.csv')
    merged_df.to_csv(output_path, index=False)

    print(f'Saved {len(files)} files for {station_code} -> {output_path}')




In [ ]:
# Load the created merged files and build a timestamp column
merged_files = sorted(glob.glob(os.path.join(merged_dir, '*_merged.csv')))

for merged_file in merged_files:
    df = pd.read_csv(merged_file)

    required_cols = ['Year', 'Month', 'Day', 'Hour', 'Minute']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f'Skipping {merged_file}: missing columns {missing_cols}')
        continue

    df['timestamp'] = pd.to_datetime(
        df[['Year', 'Month', 'Day', 'Hour', 'Minute']].astype(str).agg(lambda r: '-'.join(r[['Year', 'Month', 'Day']]) + ' ' + ':'.join(r[['Hour', 'Minute']]), axis=1),
        format='%Y-%m-%d %H:%M',
        errors='coerce'
    )
    df = df.drop(columns=required_cols)

    df.to_csv(merged_file, index=False)
    print(f'Updated {merged_file} with timestamp column')

In [ ]:
"""Filter CYL_GHI records to 2017Ã¢â‚¬â€œ2019 and fill mapped missing measurements from NSRDB by timestamp.

Input: CYL_GHI CSVs and merged station CSVs. Output: filtered station CSVs in CYL_GHI/filtered_files.
"""
# Filter CYL_GHI files to 2017-2019 and impute missing values from merged NSRDB data
cyl_ghi_dir = os.path.join(base_dir, 'CYL_GHI')
filtered_dir = os.path.join(cyl_ghi_dir, 'filtered_files')
os.makedirs(filtered_dir, exist_ok=True)

column_map = {
    'GHI': 'GHI',
    'wind_dir': 'Wind Direction',
    'humidity': 'Relative Humidity',
    'air_temp': 'Temperature',
    'wind_sp': 'Wind Speed',
    'sun_elev': 'solar_elev',
}

for file_path in sorted(glob.glob(os.path.join(cyl_ghi_dir, '*.csv'))):
    df = pd.read_csv(file_path)

    df = df[(df['timestamp'] >= '2017-01-01') & (df['timestamp'] <= '2019-12-31 23:59:59')]

    station_code = os.path.splitext(os.path.basename(file_path))[0].split('_')[0]
    merged_station_path = os.path.join(merged_dir, f'{station_code}_merged.csv')

    if os.path.exists(merged_station_path):
        merged_df = pd.read_csv(merged_station_path, parse_dates=['timestamp'])
        merged_df['timestamp'] = pd.to_datetime(merged_df['timestamp'], errors='coerce')
        merged_df = merged_df.set_index('timestamp')
        df = df.set_index('timestamp')

        # Validate mapped columns exist in both dataframes and report missing ones
        missing_in_df = [c for c in column_map.keys() if c not in df.columns]
        missing_in_merged = [c for c in column_map.values() if c not in merged_df.columns]
        if missing_in_df:
            print(f"Warning: missing columns in {os.path.basename(file_path)}: {missing_in_df}")
        if missing_in_merged:
            print(f"Warning: missing columns in merged file {merged_station_path}: {missing_in_merged}")

        for source_col, merged_col in column_map.items():
            if source_col in df.columns and merged_col in merged_df.columns:
                fill_values = merged_df[merged_col].reindex(df.index)
                df[source_col] = df[source_col].mask(
                    df[source_col].isna() | (df[source_col].astype(str).str.strip() == ''),
                    fill_values
                )

        df = df.reset_index()
    else:
        print(f'No merged station file found for {station_code}: {merged_station_path}')

    # Fill precipitation and toa from each station's own time series, not NSRDB.
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
        df = df.sort_values('timestamp').set_index('timestamp')
        for col in ['precipitation', 'toa']:
            if col in df.columns:
                values = pd.to_numeric(df[col], errors='coerce')
                values = values.interpolate(method='time', limit_area='inside').ffill().bfill()
                df[col] = values
        df = df.reset_index()

    filtered_path = os.path.join(filtered_dir, f'{station_code}_filtered.csv')
    df.to_csv(filtered_path, index=False)
    print(f'Filtered and saved {filtered_path} ({len(df)} rows)')

In [ ]:
"""Join ERA5 total cloud cover (tcc) onto filtered station files using timestamp.

Input: filtered CSVs and matching Cloud_era5 CSVs. Output: updated filtered CSVs with tcc where timestamps match.
"""
# Join TCC column from Cloud_era5 files to filtered files based on timestamp and station name
cloud_era5_dir = os.path.join(base_dir, 'Cloud_era5')

for file_path in sorted(glob.glob(os.path.join(filtered_dir, '*.csv'))):
    df = pd.read_csv(file_path, parse_dates=['timestamp'])
    
    # Extract station name from filename
    station_name = os.path.splitext(os.path.basename(file_path))[0].split('_')[0]
    
    # Find matching Cloud_era5 file
    cloud_file_pattern = os.path.join(cloud_era5_dir, f'{station_name}*.csv')
    cloud_files = glob.glob(cloud_file_pattern)
    
    if cloud_files:
        cloud_df = pd.read_csv(cloud_files[0], parse_dates=['valid_time'])
        
        # Merge on timestamp from filtered files and valid_time from Cloud_era5 files
        df = pd.merge(df, cloud_df[['valid_time', 'tcc']], left_on='timestamp', right_on='valid_time', how='left')
        df.drop(columns=['valid_time'], inplace=True)
        
        # Save the updated file
        df.to_csv(file_path, index=False)
        print(f'Added TCC column to {os.path.basename(file_path)}')
    else:
        print(f'No Cloud_erra5 file found for {station_name}')

In [ ]:
"""Report remaining blank or missing values in each filtered station CSV; this cell does not modify files.
"""
# Set either bound to None to leave that side of the period open.
analysis_start = '2017-01-01 00:00:00'
analysis_end = '2019-12-31 23:59:59'

csv_files = sorted(glob.glob(os.path.join(filtered_dir, '*.csv')))

for file_path in csv_files:
    df = pd.read_csv(file_path, parse_dates=['timestamp'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

    in_period = pd.Series(True, index=df.index)
    if analysis_start is not None:
        in_period &= df['timestamp'] >= pd.Timestamp(analysis_start)
    if analysis_end is not None:
        in_period &= df['timestamp'] <= pd.Timestamp(analysis_end)
    df = df.loc[in_period]

    blank_mask = df.isna() | df.astype(str).apply(lambda col: col.str.strip() == '')
    blank_columns = [col for col in df.columns if blank_mask[col].any()]

    if blank_columns:
        blank_counts = {col: int(blank_mask[col].sum()) for col in blank_columns}
        print(f"{os.path.basename(file_path)}: blank values found in {blank_counts}")
    else:
        print(f"{os.path.basename(file_path)}: no blank values found in selected period ({len(df)} rows)")

In [ ]:
"""Add sine/cosine encodings for wind direction, solar angles, and time of day.

Input and output: filtered station CSVs, updated in place with cyclic feature columns.
"""
# Add sinusoidal features for wind direction, sun elevation, sun azimuth, and time of day
import numpy as np

for file_path in sorted(glob.glob(os.path.join(filtered_dir, '*.csv'))):
    df = pd.read_csv(file_path, parse_dates=['timestamp'])

    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

    for col in ['wind_dir', 'sun_elev', 'sun_azim']:
        if col in df.columns:
            radians = np.radians(df[col].astype(float))
            df[f'{col}_sin'] = np.sin(radians)
            df[f'{col}_cos'] = np.cos(radians)

    if 'timestamp' in df.columns:
        seconds = (
            df['timestamp'].dt.hour * 3600
            + df['timestamp'].dt.minute * 60
            + df['timestamp'].dt.second
        )
        tod_radians = 2 * np.pi * seconds / 86400
        df['tod_sin'] = np.sin(tod_radians)
        df['tod_cos'] = np.cos(tod_radians)

    df.to_csv(file_path, index=False)
    print(f'Updated {os.path.basename(file_path)} with sinusoidal features')

In [ ]:
"""Build or reload the station tensor and save it with feature names and normalization statistics.

The tensor has shape [T, N, F]: time, station, feature. Rebuilding appends z-score normalized versions of selected measurements to the original features.
"""
# --------------------------------------------------------
# Toggle whether to reuse an existing .npz tensor or rebuild it
# --------------------------------------------------------
load_existing_npz = False  # Set to True to load an existing .npz file, False to rebuild from filtered CSVs

# --------------------------------------------------------
# Inclusive training period used to fit normalization metrics
# --------------------------------------------------------
training_start = "2017-01-01 00:00:00"
training_end = "2018-12-31 23:59:59"

# --------------------------------------------------------
# Output path
# --------------------------------------------------------
tensor_path = os.path.join(filtered_dir, "filtered_tensor.npz")

if load_existing_npz and os.path.exists(tensor_path):
    with np.load(tensor_path, allow_pickle=True) as data:
        tensor = data["tensor"]
        features = data["features"]
        feature_means = data["feature_means"]
        feature_stds = data["feature_stds"]
        normalized_features = data["normalized_features"]

    print(f"Loaded existing tensor from {tensor_path}")
    print(f"Tensor shape : {tensor.shape}")
    print(f"Loaded features : {features}")
else:
    # --------------------------------------------------------
    # Read station files
    # --------------------------------------------------------
    filtered_files = sorted(glob.glob(os.path.join(filtered_dir, "*.csv")))
    if not filtered_files:
        raise FileNotFoundError(f"No filtered CSV files found in {filtered_dir}")

    # --------------------------------------------------------
    # Fields to normalize with z-score
    # --------------------------------------------------------
    desired_features = [
        "GHI",
        "toa",
        "humidity",
        "precipitation",
        "air_temp",
        "wind_sp"
    ]

    # --------------------------------------------------------
    # Read all CSVs
    # --------------------------------------------------------
    dfs = []

    reference_columns = None
    reference_length = None
    reference_timestamps = None

    for file_path in filtered_files:

        df = pd.read_csv(file_path)

        if "timestamp" not in df.columns:
            raise ValueError(f"Missing timestamp column in {os.path.basename(file_path)}")
        timestamps = pd.to_datetime(df["timestamp"], errors="coerce")
        if timestamps.isna().any():
            raise ValueError(f"Invalid timestamps found in {os.path.basename(file_path)}")
        if reference_timestamps is None:
            reference_timestamps = timestamps.reset_index(drop=True)
        elif not timestamps.reset_index(drop=True).equals(reference_timestamps):
            raise ValueError(f"Timestamp mismatch detected in {os.path.basename(file_path)}")
        df = df.drop(columns="timestamp")

        # ----------------------------------------------------
        # Check feature consistency
        # ----------------------------------------------------
        if reference_columns is None:
            reference_columns = list(df.columns)
        elif list(df.columns) != reference_columns:
            raise ValueError(
                f"\nFeature mismatch detected.\n"
                f"File: {os.path.basename(file_path)}\n"
                f"Expected:\n{reference_columns}\n"
                f"Found:\n{list(df.columns)}"
            )

        # ----------------------------------------------------
        # Check timestep consistency
        # ----------------------------------------------------
        if reference_length is None:
            reference_length = len(df)
        elif len(df) != reference_length:
            raise ValueError(
                f"\nTimestep mismatch detected.\n"
                f"Expected {reference_length} rows, "
                f"found {len(df)} rows."
            )

        dfs.append(df)

    # --------------------------------------------------------
    # Verify desired features exist
    # --------------------------------------------------------
    missing = [f for f in desired_features if f not in reference_columns]

    if missing:
        raise ValueError(
            "The following desired features are missing:\n"
            f"{missing}"
        )

    # --------------------------------------------------------
    # Fit z-score metrics using training timestamps only
    # The same metrics are then applied to every timestamp.
    # --------------------------------------------------------
    training_start = pd.Timestamp(training_start)
    training_end = pd.Timestamp(training_end)
    if training_start > training_end:
        raise ValueError("training_start must be less than or equal to training_end")

    training_mask = (reference_timestamps >= training_start) & (reference_timestamps <= training_end)
    if not training_mask.any():
        raise ValueError("The configured training period contains no timestamps")

    feature_means = {}
    feature_stds = {}

    for feature in desired_features:
        values = pd.concat(
            [pd.to_numeric(df.loc[training_mask, feature], errors="coerce") for df in dfs],
            ignore_index=True,
        )
        feature_means[feature] = values.mean()
        std_value = values.std(ddof=0)
        feature_stds[feature] = std_value if pd.notna(std_value) and std_value > 0 else np.nan

    # --------------------------------------------------------
    # Output feature names
    # Original features + normalized feature variants
    # --------------------------------------------------------
    normalized_feature_names = [f"{f}_norm" for f in desired_features]

    output_features = (
        reference_columns +
        normalized_feature_names
    )

    # --------------------------------------------------------
    # Tensor dimensions
    # --------------------------------------------------------
    T = reference_length
    N = len(dfs)
    F_original = len(reference_columns)
    F_norm = len(desired_features)
    F_total = F_original + F_norm

    tensor = np.full(
        (T, N, F_total),
        np.nan,
        dtype=float,
    )

    # --------------------------------------------------------
    # Build tensor
    # --------------------------------------------------------
    for station_idx, df in enumerate(dfs):

        # Original features
        for feature_idx, feature in enumerate(reference_columns):

            tensor[:, station_idx, feature_idx] = pd.to_numeric(
                df[feature],
                errors="coerce",
            )

        # Normalized feature variants
        for norm_idx, feature in enumerate(desired_features):

            arr = pd.to_numeric(
                df[feature],
                errors="coerce",
            ).to_numpy(dtype=float)

            mean_val = feature_means[feature]
            std_val = feature_stds[feature]

            if np.isfinite(mean_val) and np.isfinite(std_val) and std_val > 0:
                arr = (arr - mean_val) / std_val
            elif np.isfinite(mean_val):
                arr = np.zeros_like(arr)
            else:
                arr = np.full_like(arr, np.nan)

            tensor[:, station_idx, F_original + norm_idx] = arr

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------
    np.savez_compressed(
        tensor_path,
        tensor=tensor,
        features=np.array(output_features, dtype=object),
        feature_means=np.array(
            [feature_means[f] for f in desired_features],
            dtype=float,
        ),
        feature_stds=np.array(
            [feature_stds[f] for f in desired_features],
            dtype=float,
        ),
        normalized_features=np.array(
            desired_features,
            dtype=object,
        ),
    )

    print(f"Saved tensor to {tensor_path}")
    print(f"Tensor shape : {tensor.shape}")
    print(f"Original features   : {F_original}")
    print(f"Normalized features : {F_norm}")
    print(f"Total features      : {F_total}")
import os
import glob
import pandas as pd

In [ ]:

"""Inspect the saved tensor and display the feature values for one selected station.

Input: filtered_tensor.npz. Change station_idx to inspect another station; this cell does not alter the archive.
"""
if not os.path.exists(tensor_path):
    raise FileNotFoundError(f"Tensor file not found: {tensor_path}")

with np.load(tensor_path, allow_pickle=True) as data:
    tensor = data["tensor"]
    features = data["features"].astype(str)

print(f"tensor shape: {tensor.shape}")
print("features:", list(features))

station_idx = 0  # change to view another station
if not (0 <= station_idx < tensor.shape[1]):
    raise IndexError("station_idx out of range")

pd.set_option('display.max_columns', None)

df_station = pd.DataFrame(tensor[:, station_idx, :], columns=features)
print(df_station.head())

In [ ]:
"""Create in-memory input and target views from the full tensor using named feature selections.

filtered_tensor selects model inputs; target_tensor selects raw GHI.
To recreate the target later, load tensor and features from filtered_tensor.npz and 
use tensor[:, :, features.astype(str) == 'GHI']; this yields [T, N, 1]. 
The loader can use that boolean feature mask as output_mask_key.
"""
#--------------------------------------------------------
# Filter tensor to selected features (input and target features)
#--------------------------------------------------------
selected_features = ['GHI_norm','toa_norm','humidity_norm', 'precipitation_norm',
                     'air_temp_norm','wind_sp_norm','tcc','wind_dir_sin','wind_dir_cos',
                     'sun_elev_sin','sun_elev_cos','sun_azim_sin','sun_azim_cos','tod_sin','tod_cos']

if 'tensor' not in globals() or 'features' not in globals():
    with np.load(tensor_path, allow_pickle=True) as data:
        tensor = data['tensor']
        features = data['features'].astype(str)

input_mask = np.isin(features, selected_features)
filtered_tensor = tensor[:, :, input_mask]
filtered_features = features[input_mask]

if 'GHI' in features:
    target_mask = features == 'GHI'
else:
    raise ValueError('GHI not found in tensor features')

target_tensor = tensor[:, :, target_mask]

print('filtered_tensor shape:', filtered_tensor.shape)
print(filtered_features)
print('target_tensor shape:', target_tensor.shape)